# 📱 SMS Spam Classifier — Interactive Walkthrough
**HTB Academy: Applications of AI in InfoSec**
*Author: lyethar (Fabian)*

This notebook walks through every step of building an SMS spam classifier with full explanations and red-team analysis at each stage.

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
import joblib
import warnings
warnings.filterwarnings('ignore')
print('Libraries imported successfully')

## Step 2 — Load the Dataset

The UCI SMS Spam Collection contains 5,574 labelled SMS messages.

> See `data/README.md` for download instructions. Place the file at `data/spam.csv`.

In [ ]:
df = pd.read_csv('../data/spam.csv', sep='\t', names=['label', 'message'], encoding='latin-1')
print(f'Total: {len(df)} | Spam: {len(df[df.label=="spam"])} | Ham: {len(df[df.label=="ham"])}')
df.head(10)

## Step 3 — Explore the Data

In [ ]:
df['msg_length'] = df['message'].apply(len)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'salmon'])
axes[0].set_title('Class Distribution'); axes[0].tick_params(rotation=0)
df.groupby('label')['msg_length'].plot(kind='hist', bins=50, alpha=0.7, ax=axes[1])
axes[1].set_title('Message Length by Class'); axes[1].legend(['ham', 'spam'])
plt.tight_layout(); plt.show()
print(f'Avg spam length: {df[df.label=="spam"]["msg_length"].mean():.0f} | Avg ham length: {df[df.label=="ham"]["msg_length"].mean():.0f}')

## Step 4 — Encode Labels & Split Data

Convert `spam → 1` and `ham → 0`. Split 80% train / 20% test with stratification.

In [ ]:
df['label_encoded'] = df['label'].map({'spam': 1, 'ham': 0})
X, y = df['message'], df['label_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

## Step 5 — Build TF-IDF + Naive Bayes Pipeline

**TF-IDF**: Converts words to numbers. Rewards words that are distinctive to a message, penalises common filler words.

**Naive Bayes**: Calculates P(spam | words) using Bayes' theorem. Fast, interpretable, excellent on text.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words='english',
        max_features=5000,
        ngram_range=(1, 2)  # Single words AND word pairs
    )),
    ('classifier', MultinomialNB(alpha=0.1))
])
pipeline.fit(X_train, y_train)
print('Model trained successfully')

## Step 6 — Evaluate the Model

In [ ]:
y_pred = pipeline.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'Precision: {precision_score(y_test, y_pred)*100:.2f}%')
print(f'Recall: {recall_score(y_test, y_pred)*100:.2f}%')
print(f'F1-Score: {f1_score(y_test, y_pred)*100:.2f}%')
print()
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Ham', 'Predicted Spam'],
            yticklabels=['Actual Ham', 'Actual Spam'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f'True Neg: {tn} | False Pos: {fp} | False Neg (spam missed): {fn} | True Pos: {tp}')

## Step 7 — 🔴 Red Team: Feature Importance Analysis

By extracting which words carry the highest spam weight, we simulate how an attacker would probe this model to craft evasion attacks.

In [ ]:
vectoriser = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['classifier']
feature_names = vectoriser.get_feature_names_out()
spam_score = classifier.feature_log_prob_[1] - classifier.feature_log_prob_[0]
top_spam_idx = np.argsort(spam_score)[-15:][::-1]
top_ham_idx = np.argsort(spam_score)[:15]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
spam_words = [feature_names[i] for i in top_spam_idx]
axes[0].barh(spam_words[::-1], [spam_score[i] for i in top_spam_idx][::-1], color='salmon')
axes[0].set_title('Top Spam Indicator Words')
ham_words = [feature_names[i] for i in top_ham_idx]
axes[1].barh(ham_words[::-1], [spam_score[i] for i in top_ham_idx][::-1], color='steelblue')
axes[1].set_title('Top Ham Indicator Words')
plt.tight_layout(); plt.show()
print('RED TEAM: An attacker would avoid/obfuscate all words in the spam chart above.')

## Step 8 — Try It Yourself

In [ ]:
def classify(message):
    proba = pipeline.predict_proba([message])[0]
    label = 'SPAM' if proba[1] > 0.5 else 'HAM'
    print(f'{label} ({proba[1]*100:.1f}% spam) | {message[:70]}')

classify('WINNER!! FREE prize claim your cash reward NOW! Urgent!')
classify('Hey, are we still on for lunch tomorrow?')
classify('Congratulations! You have been selected for a cash prize.')
classify("I'll be home by 6, can you start dinner?")

## Step 9 — Save the Model

In [ ]:
import os
os.makedirs('../model', exist_ok=True)
joblib.dump(pipeline, '../model/spam_classifier.pkl')
print('Model saved to model/spam_classifier.pkl')